# Lab 3 — Data Visualization and Exploratory Data Analysis

**Course:** Data Mining Lab  
**Notebook:** Lab 3 — Data Visualization and Exploratory Data Analysis

## Objectives
- Create histograms, bar charts, scatter plots, box plots and a correlation heatmap.
- Explore outliers and relationships between variables.

> Run cells from top to bottom. The notebook uses a reproducible random seed. For Adult Income labs, it attempts to load the public UCI Adult dataset and uses a clearly labelled local synthetic fallback if the download is unavailable.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
COLUMNS = ["age","workclass","fnlwgt","education","education.num","marital.status",
           "occupation","relationship","race","sex","capital.gain","capital.loss",
           "hours.per.week","native.country","income"]

def load_adult():
    local = Path("adult.csv")
    if local.exists():
        data = pd.read_csv(local)
        if set(COLUMNS).issubset(data.columns):
            return data[COLUMNS].copy(), "local adult.csv"
    try:
        data = pd.read_csv(DATA_URL, names=COLUMNS, na_values="?", skipinitialspace=True)
        return data, "UCI Adult dataset (downloaded)"
    except Exception as exc:
        rng = np.random.default_rng(42)
        n = 1200
        age = rng.integers(18, 75, n)
        hours = np.clip(rng.normal(40, 12, n).round(), 1, 80).astype(int)
        edu = rng.integers(1, 16, n)
        gain = rng.choice([0,0,0,0,500,1500,5000,10000], n)
        score = -5 + 0.055*age + 0.10*edu + 0.025*hours + (gain>0)*1.0
        income = np.where(rng.random(n) < 1/(1+np.exp(-score)), ">50K", "<=50K")
        data = pd.DataFrame({
            "age":age,"workclass":rng.choice(["Private","Self-emp","Government"],n),
            "fnlwgt":rng.integers(20000,500000,n),
            "education":rng.choice(["HS-grad","Bachelors","Masters","Some-college"],n),
            "education.num":edu,"marital.status":rng.choice(["Married","Never-married","Divorced"],n),
            "occupation":rng.choice(["Tech","Sales","Service","Admin"],n),
            "relationship":rng.choice(["Husband","Not-in-family","Own-child"],n),
            "race":rng.choice(["White","Black","Other"],n),"sex":rng.choice(["Male","Female"],n),
            "capital.gain":gain,"capital.loss":rng.choice([0,0,0,500,1000],n),
            "hours.per.week":hours,"native.country":rng.choice(["United-States","Other"],n),
            "income":income})
        print("NOTICE: Public dataset unavailable; using reproducible synthetic teaching data. Reason:", str(exc)[:180])
        return data, "synthetic fallback (not UCI observations)"

df, DATA_SOURCE = load_adult()
print("Data source:", DATA_SOURCE, "| Shape:", df.shape)
display(df.head())

df = df.copy()
for c in df.select_dtypes(include="object"):
    df[c] = df[c].fillna(df[c].mode().iloc[0])

## Histograms and bar charts

In [ ]:
import matplotlib.pyplot as plt
df["age"].hist(bins=20, figsize=(7,4))
plt.title("Age distribution"); plt.xlabel("Age"); plt.ylabel("Frequency"); plt.show()

df["education"].value_counts().head(10).plot(kind="bar", figsize=(9,4))
plt.title("Most common education categories"); plt.ylabel("Count"); plt.xticks(rotation=35); plt.show()

## Scatter plot and box plot

In [ ]:
plt.figure(figsize=(7,4))
plt.scatter(df["age"], df["hours.per.week"], alpha=.25)
plt.title("Age vs hours worked per week"); plt.xlabel("Age"); plt.ylabel("Hours per week"); plt.show()

plt.figure(figsize=(6,3))
plt.boxplot(df["hours.per.week"], vert=False)
plt.title("Weekly working hours — box plot"); plt.show()

## Correlation heatmap and relationships

In [ ]:
numeric = df.select_dtypes(include=np.number)
corr = numeric.corr()
plt.figure(figsize=(9,6))
plt.imshow(corr, cmap="coolwarm", aspect="auto", vmin=-1, vmax=1)
plt.colorbar(label="Correlation")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.title("Correlation heatmap"); plt.tight_layout(); plt.show()

q1, q3 = df["age"].quantile([.25,.75])
iqr = q3-q1
outliers = df[(df["age"] < q1-1.5*iqr) | (df["age"] > q3+1.5*iqr)]
print("Potential age outliers by IQR:", len(outliers))
display(pd.crosstab(df["education"], df["income"]))